In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
import time

spark = SparkSession.builder.appName("phase2-features").getOrCreate()
tx = spark.read.parquet("/home/jovyan/work/data/processed/train_transaction_clean")
print(f"loaded: {tx.count():,} rows × {len(tx.columns)} cols")

loaded: 590,540 rows × 290 cols


In [2]:
tx = tx.withColumns({
    "day":   F.floor(F.col("TransactionDT") / 86400),
    "hour":  F.floor((F.col("TransactionDT") % 86400) / 3600),
    "wd":    (F.floor(F.col("TransactionDT") / 86400) % 7),
})
print("calendar cols added →", len(tx.columns), "cols now")

calendar cols added → 293 cols now


In [3]:
t0 = time.time()
# "all transactions of THIS card in the trailing 24h (inclusive of current second)"
w24h  = Window.partitionBy("card1").orderBy("TransactionDT").rangeBetween(-86400 + 1, 0)
tx = (tx
    .withColumn("card1_txn_24h",  F.count(F.lit(1)).over(w24h))
    .withColumn("card1_amt_24h",  F.sum("TransactionAmt").over(w24h)))

# lifetime stats per card (no time order needed → no orderBy)
wLife = Window.partitionBy("card1")
tx = (tx
    .withColumn("card1_txn_all",  F.count(F.lit(1)).over(wLife))
    .withColumn("card1_amt_avg",  F.avg("TransactionAmt").over(wLife))
    .withColumn("amt_vs_card",    F.round(F.col("TransactionAmt") / F.col("card1_amt_avg"), 4)))

print(f"velocity features built in {time.time()-t0:.1f}s")
tx.select("card1", "TransactionDT", "TransactionAmt",
          "card1_txn_24h", "card1_amt_24h", "card1_txn_all", "amt_vs_card").show(5)

velocity features built in 1.1s
+-----+-------------+--------------+-------------+-------------+-------------+-----------+
|card1|TransactionDT|TransactionAmt|card1_txn_24h|card1_amt_24h|card1_txn_all|amt_vs_card|
+-----+-------------+--------------+-------------+-------------+-------------+-----------+
| 1010|      9826142|       277.932|            1|      277.932|            1|        1.0|
| 1015|      1751791|          63.6|            1|         63.6|            8|     0.6862|
| 1015|      1919379|       108.205|            1|      108.205|            8|     1.1674|
| 1015|      2128731|        31.132|            1|       31.132|            8|     0.3359|
| 1015|      3734666|        22.037|            1|       22.037|            8|     0.2378|
+-----+-------------+--------------+-------------+-------------+-------------+-----------+
only showing top 5 rows



In [4]:
tx.write.mode("overwrite").parquet("/home/jovyan/work/data/processed/train_features")
final = spark.read.parquet("/home/jovyan/work/data/processed/train_features")
final.select("day","hour","wd","card1_txn_24h","amt_vs_card","isFraud").show(5)
print(f"feature table saved: {final.count():,} rows × {len(final.columns)} cols")

+---+----+---+-------------+-----------+-------+
|day|hour| wd|card1_txn_24h|amt_vs_card|isFraud|
+---+----+---+-------------+-----------+-------+
|113|  17|  1|            1|        1.0|      0|
| 21|  21|  0|            1|     1.0345|      0|
| 22|  23|  1|            1|     0.8276|      1|
| 22|  23|  1|            2|     0.8276|      1|
| 25|  13|  4|            1|     2.4828|      0|
+---+----+---+-------------+-----------+-------+
only showing top 5 rows

feature table saved: 590,540 rows × 298 cols
